# RAG Fitness & Nutrition : chatbot évalué avec détection d'hallucinations

**Sujet 6 – TALN, filière IA & Science des Données (5e année), EMSI Rabat, 2026/2027**
Auteur(s) : *à compléter*

Ce notebook contient tout le pipeline : ingestion des PDF, trois stratégies de découpage, index vectoriel, génération avec deux modèles, détecteur d'hallucinations par inférence entre phrases (NLI), évaluation sur les questions de test, ablations et analyse des erreurs.

## Avant de lancer
1. Dans Google Drive, créer le dossier `MyDrive/rag-fitness-nutrition/` avec :
   - `data/raw/` : les PDF du corpus ;
   - `eval/questions.json` : le fichier des 60 questions ;
   - (optionnel) `data/sources.csv` : titre, auteur, année, URL, licence de chaque PDF.
2. Colab : GPU T4 recommandé (*Exécution → Modifier le type d'exécution*). Sans GPU, le notebook se réduit automatiquement (12 questions, un seul découpage pour la génération) et reste exécutable sur CPU.
3. *Exécution → Tout exécuter*. Avec GPU et `QUICK_MODE = True` (20 questions) comptez environ 15 à 20 minutes ; sur CPU, environ 30 à 40 minutes pour 12 questions. Passez `QUICK_MODE` à `False` pour les 60 questions (GPU conseillé).

Les graines sont fixées et la génération est déterministe (pas d'échantillonnage).

## 1. Installation et configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q chromadb sentence-transformers transformers pymupdf langchain-text-splitters pandas matplotlib scikit-learn

In [ ]:
import os, re, json, time, random, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---- Chemins (Google Drive) ----
PROJECT = Path("/content/drive/MyDrive/rag-fitness-nutrition")
RAW_DIR = PROJECT / "data" / "raw"
PROCESSED_DIR = PROJECT / "data" / "processed"
QUESTIONS_FILE = PROJECT / "eval" / "questions.json"
RESULTS_DIR = PROJECT / "eval" / "results"
CHROMA_DIR = "/content/chroma_db"          # index reconstruit à chaque session
for d in (PROCESSED_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- Paramètres expérimentaux ----
QUICK_MODE = True          # True : 5 questions par catégorie (20) ; False : les 60 questions
N_PER_CATEGORY = 5
TOP_K = 5
MAX_NEW_TOKENS = 300
RESUME = True              # reprend les résultats déjà sauvegardés sur Drive

EMBEDDINGS = {
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",                      # base de référence
    "multi": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",  # ablation multilingue
}
BASE_EMB = "minilm"
COLLECTION_PLAN = [("fixed", "minilm"), ("recursive", "minilm"),
                   ("semantic", "minilm"), ("recursive", "multi")]

GENERATION_MODELS = {
    "Qwen 0.5B": "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen 1.5B": "Qwen/Qwen2.5-1.5B-Instruct",
}
# Configurations de génération : modèle -> stratégies de découpage testées
GRID = {
    "Qwen 0.5B": ["recursive"],
    "Qwen 1.5B": ["recursive", "fixed", "semantic"],
}

NLI_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
ENTAILMENT_THRESHOLD = 0.85
CONTRADICTION_THRESHOLD = 0.80
MIN_MARGIN = 0.15

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", DEVICE)
if DEVICE == "cpu":
    # Mode CPU : configuration réduite pour rester exécutable (environ 30 à 40 minutes)
    GRID = {"Qwen 0.5B": ["recursive"], "Qwen 1.5B": ["recursive"]}
    N_PER_CATEGORY = 3
    MAX_NEW_TOKENS = 200
    print("Pas de GPU : configuration réduite (12 questions, découpage recursive pour la génération).")
    print("Le découpage et le modèle d'embeddings restent comparés sur le retrieval seul (section 5).")
assert RAW_DIR.exists(), f"Dossier introuvable : {RAW_DIR}"
assert QUESTIONS_FILE.exists(), f"Fichier introuvable : {QUESTIONS_FILE}"

## 2. Corpus et origine des données

Le corpus est composé de documents publics sur la nutrition sportive et l'entraînement en résistance : position stands de l'International Society of Sports Nutrition (ISSN, revues en accès libre), articles de revues en accès libre et avis de l'ANSES. Chaque fichier est conservé avec son nom, ce qui permet de citer la source et la page dans les réponses. Les URL et les licences sont indiquées dans `data/sources.csv` (affiché ci-dessous s'il existe).

In [ ]:
import fitz  # PyMuPDF

pages = []
for pdf in sorted(RAW_DIR.glob("*.pdf")):
    with fitz.open(pdf) as doc:
        for i, page in enumerate(doc, start=1):
            text = page.get_text()
            if text.strip():
                pages.append({"source": pdf.name, "page": i, "text": text})

with open(PROCESSED_DIR / "documents.json", "w", encoding="utf-8") as f:
    json.dump(pages, f, ensure_ascii=False)

df_pages = pd.DataFrame(pages)
df_pages["chars"] = df_pages["text"].str.len()
sources_table = (df_pages.groupby("source")
                 .agg(pages=("page", "count"), caracteres=("chars", "sum"))
                 .reset_index())
print(f"{sources_table.shape[0]} PDF, {len(pages)} pages extraites")
display(sources_table)

sources_csv = PROJECT / "data" / "sources.csv"
if sources_csv.exists():
    display(pd.read_csv(sources_csv))
else:
    print("data/sources.csv absent : à ajouter (titre, auteur, année, URL, licence).")

## 3. Stratégies de découpage (chunking)

Trois stratégies sont comparées, toutes appliquées page par page pour conserver le numéro de page :
1. **Fixed** : fenêtres de 500 caractères avec chevauchement de 100.
2. **Recursive** : `RecursiveCharacterTextSplitter` (taille 500, chevauchement 100), qui respecte paragraphes et phrases.
3. **Semantic** : regroupement de phrases consécutives tant que leur similarité reste ≥ 0,65 (taille maximale 500, minimale 150 caractères).

In [ ]:
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

embedders = {k: SentenceTransformer(v, device=DEVICE) for k, v in EMBEDDINGS.items()}

MIN_CHUNK_CHARS = 50

def fixed_chunks(pages, size=500, overlap=100):
    out, step = [], size - overlap
    for p in pages:
        text = " ".join(p["text"].split())
        for start in range(0, max(len(text), 1), step):
            piece = text[start:start + size]
            if len(piece.strip()) > MIN_CHUNK_CHARS:
                out.append({"source": p["source"], "page": p["page"], "text": piece})
            if start + size >= len(text):
                break
    return out

def recursive_chunks(pages, size=500, overlap=100):
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    out = []
    for p in pages:
        for piece in splitter.split_text(p["text"]):
            piece = " ".join(piece.split())
            if len(piece) > MIN_CHUNK_CHARS:
                out.append({"source": p["source"], "page": p["page"], "text": piece})
    return out

def split_sentences(text):
    text = " ".join(text.split())
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def semantic_chunks(pages, embedder, threshold=0.65, max_len=500, min_len=150):
    out = []
    for p in pages:
        sents = split_sentences(p["text"])
        flat = []
        for s in sents:
            while len(s) > max_len:
                flat.append(s[:max_len]); s = s[max_len:]
            if s:
                flat.append(s)
        if not flat:
            continue
        embs = embedder.encode(flat, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
        current, pieces = flat[0], []
        for i in range(1, len(flat)):
            sim = float(np.dot(embs[i - 1], embs[i]))
            too_long = len(current) + 1 + len(flat[i]) > max_len
            topic_change = sim < threshold and len(current) >= min_len
            if too_long or topic_change:
                pieces.append(current); current = flat[i]
            else:
                current += " " + flat[i]
        pieces.append(current)
        for piece in pieces:
            if len(piece) > MIN_CHUNK_CHARS:
                out.append({"source": p["source"], "page": p["page"], "text": piece})
    return out

CHUNKS = {
    "fixed": fixed_chunks(pages),
    "recursive": recursive_chunks(pages),
    "semantic": semantic_chunks(pages, embedders[BASE_EMB]),
}

rows = []
for name, chunks in CHUNKS.items():
    lengths = [len(c["text"]) for c in chunks]
    rows.append({"stratégie": name, "chunks": len(chunks),
                 "longueur moyenne": round(float(np.mean(lengths)), 1),
                 "longueur min": min(lengths), "longueur max": max(lengths),
                 "documents couverts": len({c["source"] for c in chunks})})
chunk_stats = pd.DataFrame(rows)
display(chunk_stats)
chunk_stats.to_csv(RESULTS_DIR / "chunking_comparison.csv", index=False)

## 4. Index vectoriel et recherche

Une collection ChromaDB (similarité cosinus) est construite par stratégie avec le modèle d'embeddings de référence `all-MiniLM-L6-v2`. Une collection supplémentaire (découpage recursive, modèle multilingue) sert à l'ablation sur le modèle d'embeddings, car les questions sont en français alors qu'une partie du corpus est en anglais.

In [ ]:
import chromadb

client = chromadb.PersistentClient(path=CHROMA_DIR)
collections = {}

def build_collection(name, chunks, embedder):
    try:
        client.delete_collection(name)
    except Exception:
        pass
    col = client.create_collection(name=name, metadata={"hnsw:space": "cosine"})
    texts = [c["text"] for c in chunks]
    embs = embedder.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    for i in range(0, len(chunks), 1000):
        j = min(i + 1000, len(chunks))
        col.add(ids=[f"{name}_{k}" for k in range(i, j)],
                embeddings=embs[i:j].tolist(),
                documents=texts[i:j],
                metadatas=[{"source": chunks[k]["source"], "page": chunks[k]["page"], "chunk_id": k}
                           for k in range(i, j)])
    return col

for strategy, emb_key in COLLECTION_PLAN:
    name = f"fitness_{strategy}_{emb_key}"
    collections[(strategy, emb_key)] = build_collection(name, CHUNKS[strategy], embedders[emb_key])
    print(f"{name}: {collections[(strategy, emb_key)].count()} vecteurs")

def retrieve(question, strategy, emb_key=BASE_EMB, k=TOP_K):
    col = collections[(strategy, emb_key)]
    q = embedders[emb_key].encode(question, normalize_embeddings=True).tolist()
    res = col.query(query_embeddings=[q], n_results=k, include=["documents", "metadatas", "distances"])
    return [{"content": d, "source": m["source"], "page": m["page"], "distance": float(dist)}
            for d, m, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])]

## 5. Jeu de questions et diagnostic du retrieval

Le jeu contient quatre catégories : réponse présente, réponse absente (refus attendu), question ambiguë et question multi-passages. Les questions marquées `removed` sont ignorées. Les métriques de retrieval fondées sur des passages de référence (`gold_sources`) ne sont calculées que pour les questions où ces références ont été renseignées.

Trois indicateurs de diagnostic ne dépendent pas des références :
- part de passages en **français** (révèle un biais de langue de l'embedding) ;
- part de passages qui ressemblent à une **liste de références bibliographiques** (heuristique) ;
- **distance moyenne** du meilleur passage.

In [ ]:
DEFAULT_BEHAVIOR = {"answer_present": "answer", "multi_passage": "answer",
                    "answer_absent": "refuse", "ambiguous": "clarify"}

with open(QUESTIONS_FILE, encoding="utf-8") as f:
    ALL_Q = [q for q in json.load(f)["questions"] if q.get("validation_status") != "removed"]

for q in ALL_Q:
    q["expected"] = q.get("expected_behavior") or DEFAULT_BEHAVIOR[q["category"]]

def select_questions(qs):
    if not QUICK_MODE:
        return qs
    by = {}
    for q in qs:
        by.setdefault(q["category"], []).append(q)
    sel = []
    for items in by.values():
        n = min(N_PER_CATEGORY, len(items))
        sel += [items[int(i * len(items) / n)] for i in range(n)]
    return sel

QUESTIONS = select_questions(ALL_Q)
QUESTION_IDS = {q["id"] for q in QUESTIONS}
n_gold = sum(1 for q in QUESTIONS if q.get("gold_sources"))
print(f"{len(QUESTIONS)} questions utilisées ({n_gold} avec passages de référence)")
display(pd.Series([q["category"] for q in QUESTIONS]).value_counts().rename("questions").to_frame())

FR_WORDS = [" le ", " la ", " les ", " des ", " une ", " est ", " pour ", " dans "]
EN_WORDS = [" the ", " of ", " and ", " with ", " that ", " is ", " for ", " in "]

def is_french(text):
    t = " " + text.lower() + " "
    return sum(t.count(w) for w in FR_WORDS) > sum(t.count(w) for w in EN_WORDS)

def looks_like_reference(text):
    return ("CrossRef" in text or text.count("doi") >= 2
            or re.search(r"\b(?:19|20)\d{2},\s*\d+,\s*\d+", text) is not None)

def gold_hit(q, passages):
    gold = q.get("gold_sources") or []
    if not gold:
        return None, None
    page_hit = any(str(p["source"]) == str(g["source"]) and str(p["page"]) == str(g["page"])
                   for p in passages for g in gold)
    doc_hit = any(str(p["source"]) == str(g["source"]) for p in passages for g in gold)
    return page_hit, doc_hit

answerable = [q for q in QUESTIONS if q["expected"] == "answer"]
rows = []
for (strategy, emb_key) in collections:
    fr, ref, dist, hits, dhits = [], [], [], [], []
    for q in answerable:
        ps = retrieve(q["question"], strategy, emb_key)
        fr.append(np.mean([is_french(p["content"]) for p in ps]))
        ref.append(np.mean([looks_like_reference(p["content"]) for p in ps]))
        dist.append(ps[0]["distance"])
        h, dh = gold_hit(q, ps)
        if h is not None:
            hits.append(h); dhits.append(dh)
    rows.append({"découpage": strategy, "embeddings": emb_key,
                 "% passages en français": round(100 * np.mean(fr), 1),
                 "% passages type références biblio": round(100 * np.mean(ref), 1),
                 "distance moyenne top-1": round(float(np.mean(dist)), 3),
                 f"hit@{TOP_K} page (n={len(hits)})": round(100 * np.mean(hits), 1) if hits else None,
                 f"hit@{TOP_K} document": round(100 * np.mean(dhits), 1) if dhits else None})
retrieval_table = pd.DataFrame(rows)
display(retrieval_table)
retrieval_table.to_csv(RESULTS_DIR / "retrieval_comparison.csv", index=False)

## 6. Détecteur d'hallucinations (NLI)

**Principe.** La réponse est découpée en affirmations. Chaque affirmation est comparée à chacun des passages récupérés par un modèle d'inférence entre phrases (`mDeBERTa-v3-base-mnli-xnli`) : *entailment* (le passage soutient), *neutral* ou *contradiction*. Une affirmation est :
- `SUPPORTED` si l'entailment maximal ≥ 0,85, avec une marge d'au moins 0,15 sur la contradiction ;
- `CONTRADICTED` si la contradiction maximale ≥ 0,80 ;
- `UNSUPPORTED` sinon.

**Règle de négation.** Une règle lexicale (détection de « ne…pas », « sans », « aucun »…) a été testée : elle classe tout en `UNSUPPORTED`. Elle est donc désactivée par défaut et conservée comme **ablation** : les deux statuts (`status` sans la règle, `status_negation_rule` avec) sont calculés à partir des mêmes scores NLI.

**Deux corrections par rapport aux scripts initiaux** : les refus ne sont pas évalués comme des affirmations, et le score de fidélité vaut `None` (et non 0 %) lorsqu'une réponse ne contient aucune affirmation évaluable.

**Limite.** Un score de fidélité élevé ne prouve pas que la réponse est correcte : le NLI réagit souvent au thème plus qu'au contenu précis (voir l'analyse des erreurs).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

nli_tok = AutoTokenizer.from_pretrained(NLI_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_NAME).to(DEVICE).eval()
ID2LABEL = {i: l.lower() for i, l in nli_model.config.id2label.items()}
ENT_ID = next(i for i, l in ID2LABEL.items() if l == "entailment")
CON_ID = next(i for i, l in ID2LABEL.items() if l == "contradiction")
print("Labels NLI :", ID2LABEL)

def clean(text):
    return " ".join(text.split())

def split_into_claims(answer):
    if not answer:
        return []
    text = re.sub(r"\[S[\d,\sS]*\]", "", answer)
    claims = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        line = re.sub(r"\*\*(.*?)\*\*", r"\1", line)
        line = re.sub(r"\*(.*?)\*", r"\1", line)
        line = re.sub(r"^\s*(?:[-*•]|\d+[.)])\s*", "", line)
        for sentence in re.split(r"(?<=[.!?])\s+", line):
            sentence = sentence.strip()
            if len(sentence.split()) < 5 or sentence[-1] not in ".!?":
                continue
            claims.append(sentence)
    return claims

NEGATION_PATTERNS = [r"\bne\b.*\bpas\b", r"\bn['’]est pas\b", r"\bpas systématiquement\b",
                     r"\bpas nécessairement\b", r"\bni\b", r"\baucun(?:e)?\b",
                     r"\babsence de\b", r"\bsans\b", r"\bnon observé\b", r"\bnon démontré\b"]

def has_negation_conflict(claim, passages):
    words = lambda s: {w.lower() for w in re.findall(r"\b[a-zA-ZÀ-ÿ]{5,}\b", s)}
    claim_words = words(claim)
    for passage in passages:
        for sentence in re.split(r"(?<=[.!?])\s+", passage["content"]):
            if claim_words & words(sentence) and any(
                    re.search(p, sentence.lower()) for p in NEGATION_PATTERNS):
                return True
    return False

def nli_scores_batch(claim, passages):
    premises = [clean(p["content"]) for p in passages]
    inputs = nli_tok(premises, [clean(claim)] * len(premises), return_tensors="pt",
                     truncation=True, max_length=512, padding=True).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(nli_model(**inputs).logits, dim=1).cpu().numpy()
    return probs[:, ENT_ID], probs[:, CON_ID]

def classify(best_ent, best_con, negation, use_rule=False,
             ent_th=None, con_th=None, margin=None):
    ent_th = ENTAILMENT_THRESHOLD if ent_th is None else ent_th
    con_th = CONTRADICTION_THRESHOLD if con_th is None else con_th
    margin = MIN_MARGIN if margin is None else margin
    if best_con >= con_th:
        return "CONTRADICTED"
    if best_ent >= ent_th and best_ent - best_con >= margin and not (use_rule and negation):
        return "SUPPORTED"
    return "UNSUPPORTED"

def check_claim(claim, passages):
    ent, con = nli_scores_batch(claim, passages)
    i = int(np.argmax(ent))
    best_ent, best_con = float(ent[i]), float(con.max())
    neg = has_negation_conflict(claim, passages)
    return {"text": claim,
            "status": classify(best_ent, best_con, neg, use_rule=False),
            "status_negation_rule": classify(best_ent, best_con, neg, use_rule=True),
            "best_entailment": round(best_ent, 4),
            "max_contradiction": round(best_con, 4),
            "negation_detected": neg,
            "evidence": {"source": passages[i]["source"], "page": passages[i]["page"],
                         "text": clean(passages[i]["content"])[:400]}}

def faithfulness(claims, key="status"):
    if not claims:
        return None
    return 100.0 * sum(c[key] == "SUPPORTED" for c in claims) / len(claims)

print("Détecteur prêt.")

## 7. Génération avec citations et refus

Le prompt impose de répondre uniquement à partir des passages, de citer `[S1]`, `[S2]`… et d'utiliser une phrase de refus fixe lorsque les documents ne permettent pas de répondre. Le refus est détecté par cette phrase. Deux modèles ouverts sont comparés (Qwen2.5 0,5B et 1,5B, génération déterministe). `MAX_NEW_TOKENS` est passé de 200 à 300 par rapport aux essais préliminaires, pour limiter les réponses tronquées ; les réponses encore tronquées sont signalées.

In [ ]:
from transformers import AutoModelForCausalLM

REFUSAL_PHRASE = "Je ne dispose pas d'informations suffisantes dans les documents pour répondre à cette question."
REFUSAL_MARKER = "ne dispose pas d'informations suffisantes"

def build_prompt(question, passages):
    context = ""
    for i, p in enumerate(passages, start=1):
        context += f"\n[S{i}] Source: {p['source']}, Page: {p['page']}\n{clean(p['content'])}\n"
    return f"""Tu es un assistant spécialisé en nutrition et activité physique.

Ta mission est de répondre à la question en utilisant exclusivement les documents fournis.

RÈGLES OBLIGATOIRES :
1. Utilise uniquement les informations des documents.
2. N'invente aucune information.
3. Ne complète pas les informations avec tes connaissances personnelles.
4. Cite les sources avec [S1], [S2], etc.
5. Si les documents ne permettent pas de répondre, réponds exactement :
   "{REFUSAL_PHRASE}"
6. Ne formule pas de recommandations médicales personnalisées.
7. Réponds en français.
8. Sois précis et concis.
9. N'ajoute aucun chiffre ou effet absent des documents.
10. Ne présente pas une hypothèse comme un fait établi.

DOCUMENTS :
{context}

QUESTION :
{question}

RÉPONSE :
"""

def load_generation_model(name):
    tok = AutoTokenizer.from_pretrained(name)
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype).to(DEVICE).eval()
    return tok, model

def generate_answer(question, passages, tok, model):
    messages = [{"role": "user", "content": build_prompt(question, passages)}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             repetition_penalty=1.1, pad_token_id=tok.eos_token_id)
    elapsed = time.time() - t0
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    answer = tok.decode(new_tokens, skip_special_tokens=True).strip()
    return answer, elapsed, len(new_tokens)

def is_refusal(answer):
    return REFUSAL_MARKER in answer.lower().replace("’", "'")

def has_citation(answer):
    return re.search(r"\[S\d+", answer) is not None

## 8. Expériences

Pour chaque configuration (modèle × stratégie de découpage) et chaque question : retrieval, génération, détection des refus, évaluation NLI des affirmations. Les résultats sont enregistrés sur Drive après chaque lot ; une session interrompue reprend où elle s'est arrêtée (`RESUME = True`).

In [ ]:
RESULTS_FILE = RESULTS_DIR / "notebook_results.json"

records = []
if RESUME and RESULTS_FILE.exists():
    with open(RESULTS_FILE, encoding="utf-8") as f:
        records = json.load(f)
done = {(r["model"], r["strategy"], r["id"]) for r in records}
print(f"{len(records)} résultats déjà présents")

def save_records():
    with open(RESULTS_FILE, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=1)

for model_label, strategies in GRID.items():
    todo = [(s, q) for s in strategies for q in QUESTIONS
            if (model_label, s, q["id"]) not in done]
    if not todo:
        continue
    print(f"\n=== {model_label} : {len(todo)} générations ===")
    tok, model = load_generation_model(GENERATION_MODELS[model_label])

    for n, (strategy, q) in enumerate(todo, start=1):
        passages = retrieve(q["question"], strategy)
        answer, g_time, n_tokens = generate_answer(q["question"], passages, tok, model)
        refused = is_refusal(answer)

        t0 = time.time()
        claims = [] if refused else [check_claim(c, passages) for c in split_into_claims(answer)]
        nli_time = time.time() - t0

        page_hit, doc_hit = gold_hit(q, passages)
        records.append({
            "id": q["id"], "category": q["category"], "expected": q["expected"],
            "question": q["question"], "model": model_label, "strategy": strategy,
            "answer": answer, "refused": refused, "has_citation": has_citation(answer),
            "truncated": n_tokens >= MAX_NEW_TOKENS,
            "generation_time": round(g_time, 2), "nli_time": round(nli_time, 2),
            "n_claims": len(claims), "claims": claims,
            "retrieval_page_hit": page_hit, "retrieval_doc_hit": doc_hit,
            "top_sources": [(p["source"], p["page"]) for p in passages],
        })
        if n % 10 == 0:
            save_records()
            print(f"  {n}/{len(todo)}")

    save_records()
    del model, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nTerminé. Résultats :", RESULTS_FILE)

## 9. Résultats

In [ ]:
df = pd.DataFrame(records)
df = df[df["id"].isin(QUESTION_IDS)].copy()
df["faith"] = df["claims"].apply(lambda c: faithfulness(c, "status"))
df["faith_rule"] = df["claims"].apply(lambda c: faithfulness(c, "status_negation_rule"))

def behavior_ok(r):
    if r["expected"] == "refuse":
        return r["refused"]
    if r["expected"] == "answer":
        return not r["refused"]
    return None          # "clarify" : à juger manuellement

df["behavior_ok"] = df.apply(behavior_ok, axis=1)

def summarize(g):
    answered = g[~g["refused"]]
    return pd.Series({
        "questions": len(g),
        "fidélité moyenne (%)": round(answered["faith"].mean(), 1),
        "contradictions (affirmations)": int(sum(c["status"] == "CONTRADICTED"
                                                 for cl in g["claims"] for c in cl)),
        "citations [S#] (%)": round(100 * answered["has_citation"].mean(), 1) if len(answered) else None,
        "réponses tronquées (%)": round(100 * g["truncated"].mean(), 1),
        "comportement attendu respecté (%)": round(100 * g["behavior_ok"].dropna().astype(float).mean(), 1),
        "temps génération (s)": round(g["generation_time"].mean(), 1),
        "temps NLI (s)": round(g["nli_time"].mean(), 2),
    })

summary = df.groupby(["model", "strategy"]).apply(summarize).reset_index()
display(summary)
summary.to_csv(RESULTS_DIR / "summary_by_config.csv", index=False)

by_cat = (df.groupby(["model", "strategy", "category"])
          .agg(questions=("id", "count"),
               taux_de_refus=("refused", "mean"),
               fidélité=("faith", "mean"))
          .round(2).reset_index())
display(by_cat)
by_cat.to_csv(RESULTS_DIR / "summary_by_category.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = [f"{m}\n{s}" for m, s in zip(summary["model"], summary["strategy"])]

axes[0].bar(labels, summary["fidélité moyenne (%)"])
axes[0].set_title("Fidélité moyenne (réponses non refusées)")
axes[0].set_ylabel("%")

axes[1].bar(labels, summary["comportement attendu respecté (%)"])
axes[1].set_title("Comportement attendu respecté (réponse / refus)")
axes[1].set_ylabel("%")

for ax in axes:
    ax.tick_params(axis="x", labelsize=8)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "results_overview.png", dpi=150)
plt.show()

## 10. Ablations et sensibilité

1. **Découpage** : comparaison des trois stratégies avec le même modèle (voir le tableau de la section 9 et le diagnostic de retrieval de la section 5).
2. **Modèle d'embeddings** : MiniLM (anglais) contre modèle multilingue, mesuré sur le retrieval (section 5).
3. **Règle de négation** : fidélité avec et sans la règle, sur les mêmes scores NLI.
4. **Seuil d'entailment** : sensibilité de la fidélité au seuil.

In [ ]:
abl = (df.groupby(["model", "strategy"])
       .agg(sans_regle=("faith", "mean"), avec_regle=("faith_rule", "mean"))
       .round(1).reset_index())
print("Ablation : règle de négation (fidélité moyenne, %)")
display(abl)
abl.to_csv(RESULTS_DIR / "ablation_negation_rule.csv", index=False)

THRESHOLDS = [0.50, 0.70, 0.85, 0.95, 0.99]
rows = []
for (model, strategy), g in df.groupby(["model", "strategy"]):
    for th in THRESHOLDS:
        vals = []
        for claims in g["claims"]:
            if not claims:
                continue
            st = [classify(c["best_entailment"], c["max_contradiction"],
                           c["negation_detected"], use_rule=False, ent_th=th)
                  for c in claims]
            vals.append(100.0 * sum(s == "SUPPORTED" for s in st) / len(st))
        rows.append({"model": model, "strategy": strategy, "seuil": th,
                     "fidélité (%)": round(float(np.mean(vals)), 1) if vals else None})
sens = pd.DataFrame(rows)
display(sens.pivot_table(index=["model", "strategy"], columns="seuil", values="fidélité (%)"))
sens.to_csv(RESULTS_DIR / "ablation_entailment_threshold.csv", index=False)

plt.figure(figsize=(7, 4))
for (model, strategy), g in sens.groupby(["model", "strategy"]):
    plt.plot(g["seuil"], g["fidélité (%)"], marker="o", label=f"{model} / {strategy}")
plt.xlabel("Seuil d'entailment"); plt.ylabel("Fidélité (%)")
plt.title("Sensibilité au seuil d'entailment"); plt.legend(fontsize=8)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "sensitivity_threshold.png", dpi=150); plt.show()

## 11. Analyse des erreurs

Les cellules suivantes affichent des cas concrets. **À commenter dans l'article** : pour chaque catégorie d'erreur, expliquer la cause probable (retrieval, génération, détecteur) à partir des exemples.

In [ ]:
base = df[(df["model"] == "Qwen 1.5B") & (df["strategy"] == "recursive")]

print("=== A. Refus attendu mais le modèle a répondu ===")
for _, r in base[(base["expected"] == "refuse") & (~base["refused"])].head(5).iterrows():
    print(f"\n[{r['id']}] {r['question']}\n  -> {r['answer'][:350]}")

print("\n\n=== B. Réponse attendue mais le modèle a refusé ===")
for _, r in base[(base["expected"] == "answer") & (base["refused"])].head(5).iterrows():
    print(f"\n[{r['id']}] {r['question']}\n  sources récupérées : {r['top_sources'][:3]}")

In [ ]:
print("=== C. Affirmations rejetées avec un entailment très faible (hallucinations probables) ===")
shown = 0
for _, r in base.iterrows():
    for c in r["claims"]:
        if c["status"] != "SUPPORTED" and c["best_entailment"] < 0.1 and shown < 6:
            print(f"\n[{r['id']}] {c['text']}\n  entailment={c['best_entailment']} | "
                  f"preuve la plus proche : {c['evidence']['source']} p.{c['evidence']['page']}")
            shown += 1

print("\n\n=== D. Affirmations validées à contrôler à la main (faux positifs possibles) ===")
shown = 0
for _, r in base.iterrows():
    for c in r["claims"]:
        if c["status"] == "SUPPORTED" and shown < 6:
            print(f"\n[{r['id']}] AFFIRMATION : {c['text']}\n  PREUVE ({c['evidence']['source']} "
                  f"p.{c['evidence']['page']}) : {c['evidence']['text'][:250]}...")
            shown += 1

## 12. Fiabilité du détecteur : annotation manuelle

Le détecteur n'a de valeur que s'il est comparé à un jugement humain. Cette cellule exporte un échantillon d'affirmations vers Drive (`eval/results/annotation_sample.csv`). **À faire à la main** : remplir la colonne `human_label` avec `SUPPORTED`, `UNSUPPORTED` ou `CONTRADICTED` en lisant la preuve, puis relancer la cellule d'évaluation en dessous.

In [ ]:
ANNOT_FILE = RESULTS_DIR / "annotation_sample.csv"

if not ANNOT_FILE.exists():
    pool = []
    for _, r in df[~df["refused"]].iterrows():
        for c in r["claims"]:
            pool.append({"id": r["id"], "model": r["model"], "strategy": r["strategy"],
                         "claim": c["text"], "evidence_source": c["evidence"]["source"],
                         "evidence_page": c["evidence"]["page"],
                         "evidence_text": c["evidence"]["text"],
                         "detector_status": c["status"], "human_label": ""})
    rng = random.Random(SEED)
    sample = []
    for status in ("SUPPORTED", "UNSUPPORTED", "CONTRADICTED"):
        group = [p for p in pool if p["detector_status"] == status]
        rng.shuffle(group)
        sample += group[:20]
    pd.DataFrame(sample).to_csv(ANNOT_FILE, index=False, encoding="utf-8-sig")
    print(f"{len(sample)} affirmations exportées vers {ANNOT_FILE}")
else:
    print("Fichier d'annotation déjà présent :", ANNOT_FILE)

In [ ]:
from sklearn.metrics import confusion_matrix, cohen_kappa_score, classification_report

ann = pd.read_csv(ANNOT_FILE)
ann = ann[ann["human_label"].astype(str).str.strip().isin(["SUPPORTED", "UNSUPPORTED", "CONTRADICTED"])]

if len(ann) < 10:
    print(f"{len(ann)} affirmations annotées : remplir la colonne human_label (au moins 30) puis relancer.")
else:
    y_true, y_pred = ann["human_label"].str.strip(), ann["detector_status"]
    labels = ["SUPPORTED", "UNSUPPORTED", "CONTRADICTED"]
    print(f"{len(ann)} affirmations annotées")
    print("Kappa de Cohen (détecteur vs humain) :", round(cohen_kappa_score(y_true, y_pred), 3))
    display(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels),
                         index=[f"humain {l}" for l in labels],
                         columns=[f"détecteur {l}" for l in labels]))
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

## 13. Limites, éthique et conclusion

**À rédiger dans l'article à partir des résultats ci-dessus :**
- **Corpus** : taille, langues (français/anglais), qualité d'extraction des PDF (listes de références, tableaux), licences des sources.
- **Retrieval** : biais de langue des embeddings (questions en français, documents surtout en anglais), effet du découpage.
- **Génération** : modèles de petite taille, réponses tronquées, respect des citations `[S#]`, refus justifiés ou non.
- **Détecteur NLI** : faux positifs lorsque l'affirmation reprend le thème d'un passage sans que celui-ci la soutienne ; sensibilité aux seuils ; règle de négation inutilisable. Un score de fidélité élevé ne garantit pas l'exactitude.
- **Jeu de questions** : taille, catégories « ambiguës » jugées manuellement, références renseignées pour un sous-ensemble.
- **Éthique** : le domaine touche à la santé. Les réponses ne sont pas des conseils médicaux ; un cas d'effets indésirables graves de compléments protéinés (avis ANSES 2025) rappelle le risque d'une réponse erronée. Les refus pour questions médicales personnalisées sont volontaires.

*Références : à compléter avec la liste des sources du corpus (voir `data/sources.csv`) et les modèles utilisés (Qwen2.5, all-MiniLM-L6-v2, paraphrase-multilingual-MiniLM-L12-v2, mDeBERTa-v3-base-mnli-xnli).*